# Exploratory Data Analysis (EDA) - Email/SMS Spam Classifier

This notebook explores the SMS Spam Collection dataset to understand:
1. Class distributions (Spam vs. Ham/Legitimate).
2. Message length and word count distributions.
3. Most frequent keywords in spam vs. ham messages.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Add project root to path for importing local modules
sys.path.append(os.path.dirname(os.path.abspath('')))
from src.preprocess import load_and_clean_data, preprocess_message

data_path = '../data/spam.csv'
if not os.path.exists(data_path):
    # If relative path is different depending on where notebook runs
    data_path = '../../email-spam-classifier/data/spam.csv'

df = load_and_clean_data(data_path)
df.head()

### 1. Class Distribution

Let's visualize the ratio of Spam vs. Ham in the dataset. Since spam represents a minority, we have a significant class imbalance.

In [ ]:
plt.figure(figsize=(7, 5))
sns.set_theme(style="whitegrid")

colors = ['#10b981', '#ef4444'] # Emerald green for Ham, Crimson red for Spam
ax = sns.countplot(x='label', data=df, palette=colors, hue='label', legend=False)
plt.title('Distribution of Ham vs. Spam Messages', fontsize=14, weight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Add percentage annotations on top of the bars
total = len(df)
for p in ax.patches:
    height = p.get_height()
    percentage = (height / total) * 100
    ax.annotate(f'{height}\n({percentage:.1f}%)',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', fontsize=11, weight='bold', xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

### 2. Message Length Analysis

Hypothesis: Spam messages tend to be longer and more descriptive (containing links, urgent calls to action, etc.) than legitimate messages.

In [ ]:
# Compute statistics
df['char_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print("=== Character Length Stats by Class ===")
print(df.groupby('label')['char_length'].describe().round(1))

print("\n=== Word Count Stats by Class ===")
print(df.groupby('label')['word_count'].describe().round(1))

#### Message Length Distribution

Let's plot character length and word count distributions for Ham vs. Spam.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Character Length
sns.histplot(data=df, x='char_length', hue='label', kde=True, bins=50,
             palette={'ham': '#10b981', 'spam': '#ef4444'}, ax=axes[0], multiple='dodge')
axes[0].set_title('Distribution of Character Length', fontsize=13, weight='semibold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Density')
axes[0].set_xlim(0, 300) # Limit x-axis to zoom in on the main distribution area

# Word Count
sns.histplot(data=df, x='word_count', hue='label', kde=True, bins=50,
             palette={'ham': '#10b981', 'spam': '#ef4444'}, ax=axes[1], multiple='dodge')
axes[1].set_title('Distribution of Word Count', fontsize=13, weight='semibold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Density')
axes[1].set_xlim(0, 70)

plt.tight_layout()
plt.show()

### 3. Most Common Words (Preprocessed)

Let's preprocess the messages (remove punctuation, stopwords, and perform stemming) and extract the most frequent words in Spam vs. Ham.

In [ ]:
from collections import Counter

print("Preprocessing all messages (this might take a few seconds)... ")
df['processed_message'] = df['message'].apply(preprocess_message)

# Get all words from ham
ham_words = []
for msg in df[df['label'] == 'ham']['processed_message']:
    ham_words.extend(msg.split())
    
# Get all words from spam
spam_words = []
for msg in df[df['label'] == 'spam']['processed_message']:
    spam_words.extend(msg.split())

ham_freq = Counter(ham_words).most_common(20)
spam_freq = Counter(spam_words).most_common(20)

df_ham_freq = pd.DataFrame(ham_freq, columns=['Word', 'Count'])
df_spam_freq = pd.DataFrame(spam_freq, columns=['Word', 'Count'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ham Common Words
sns.barplot(x='Count', y='Word', data=df_ham_freq, ax=axes[0],
            palette=sns.color_palette("crest", n_colors=20), hue='Word', legend=False)
axes[0].set_title('Top 20 Words in Ham (Legit) Messages', fontsize=13, weight='semibold')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Word')

# Spam Common Words
sns.barplot(x='Count', y='Word', data=df_spam_freq, ax=axes[1],
            palette=sns.color_palette("flare", n_colors=20), hue='Word', legend=False)
axes[1].set_title('Top 20 Words in Spam Messages', fontsize=13, weight='semibold')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Word')

plt.tight_layout()
plt.show()

### 4. Conclusion
- **Class Imbalance**: Legitimate messages make up roughly 86-88% of the dataset, whereas spam makes up only about 12-14%. We must account for this in model evaluation by prioritizing the F1-Score rather than Accuracy.
- **Length difference**: Spam messages are significantly longer on average (mean of ~137 chars vs ~70 chars for ham). This suggests that message length is a highly descriptive feature.
- **Word frequency difference**: Words like *free*, *txt*, *call*, *claim*, *mobil*, *prize*, and *stop* are highly frequent in spam messages, reflecting marketing, phishing, and scamming patterns.